# Logit GAM — pyGAM Cloud Train

**Corre en Kaggle Notebooks** (CPU, ~9h disponibles).

**Dataset requerido:** `mecmt07-features` (subir `features_train.parquet` y `features_test.parquet`).

**Modelo:** `LogisticGAM` de pyGAM — Regresión logística aditiva con términos suaves por variable:

$$p(X_1, \ldots, X_p) = \sigma\bigl(\beta_0 + g_1(X_1) + \cdots + g_p(X_p)\bigr)$$

- Features **continuas**: término suave `s(i, n_splines=10, spline_order=3)`
- Features **binarias**: término lineal `l(i)`
- Lambda seleccionado por **5-fold StratifiedKFold CV** sobre muestra completa
- Guarda: `pygam_cloud_best.pkl` y `logit_splines_cloud_metadata.json`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pygam', '--quiet'], check=True)
print('Dependencias listas.')

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import platform
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import sklearn

from pygam import LogisticGAM, s, l

print(f'sklearn version : {sklearn.__version__}')
print('Imports OK')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════
SEED         = 42
N_FOLDS      = 5
N_SPLINES    = 10
SPLINE_ORDER = 3
LAM_GRID     = np.logspace(-2, 1, 3)   # 3 lambdas → 15 fits (5×3): [0.01, 0.1, 10]

DATA_DIR  = Path('/kaggle/input/datasets/davidguzzi/mecmt07-features')
MODEL_DIR = Path('/kaggle/working')

# Matplotlib
plt.rcParams['figure.dpi']          = 100
plt.rcParams['axes.spines.top']     = False
plt.rcParams['axes.spines.right']   = False

np.random.seed(SEED)

print('=' * 65)
print('CONFIGURACIÓN — Logit GAM (pyGAM)')
print('=' * 65)
print(f'  DATA_DIR     : {DATA_DIR}')
print(f'  MODEL_DIR    : {MODEL_DIR}')
print(f'  N_FOLDS      : {N_FOLDS}')
print(f'  LAM_GRID     : {len(LAM_GRID)} valores ({LAM_GRID[0]:.1e} – {LAM_GRID[-1]:.1e})')
print(f'  N_SPLINES    : {N_SPLINES}')
print(f'  SPLINE_ORDER : {SPLINE_ORDER}')
print(f'  SEED         : {SEED}')
print('=' * 65)
for f in ['features_train.parquet', 'features_test.parquet']:
    exists = (DATA_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "NO encontrado"}')

In [ ]:
print('Cargando datos...')
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

# Encodear columnas categóricas
cat_cols = [c for c in feature_cols if df[c].dtype == 'object']
if cat_cols:
    print(f'  Columnas categóricas encontradas: {cat_cols}')
    for col in cat_cols:
        cats    = pd.concat([df[col], df_test[col]]).dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df[col]      = df[col].map(mapping)
        df_test[col] = df_test[col].map(mapping)
    print('  Encoding completado ✓')
else:
    print('  Sin columnas categóricas')

X_raw       = df[feature_cols].values
y           = df['TARGET'].values
X_test_raw  = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'\nTrain: {X_raw.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test_raw.shape}')

In [ ]:
# ── Imputación ──────────────────────────────────────────────────────────────────────────
print('Imputando NaN con mediana...')
imputer    = SimpleImputer(strategy='median')
X_imp      = imputer.fit_transform(X_raw)
X_test_imp = imputer.transform(X_test_raw)
print(f'  X_imp: {X_imp.shape} — NaNs: {np.isnan(X_imp).sum()}')

# ── Identificar features binarias (≤ 2 valores únicos) ───────────────────────────
n_unique     = np.array([len(np.unique(col[~np.isnan(col)])) for col in X_raw.T])
binary_mask  = n_unique <= 2
binary_feats = [feature_cols[i] for i in range(len(feature_cols)) if binary_mask[i]]
cont_feats   = [feature_cols[i] for i in range(len(feature_cols)) if not binary_mask[i]]
cont_idx     = [i for i, c in enumerate(feature_cols) if c in set(cont_feats)]
bin_idx      = [i for i, c in enumerate(feature_cols) if c in set(binary_feats)]

print(f'  Features binarias  ({len(binary_feats)}): {binary_feats}')
print(f'  Features continuas ({len(cont_feats)}): primeras 5: {cont_feats[:5]}')

# ── Escalado ────────────────────────────────────────────────────────────────────────
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X_imp)
X_test_scaled = scaler.transform(X_test_imp)
print(f'  X_scaled: {X_scaled.shape}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc, 4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec, 4), Precision=round(prec, 4), F1=round(f1, 4))

print('Funciones auxiliares definidas.')

---
## LogisticGAM (pyGAM)

Construcción de la fórmula GAM con términos independientes por variable.

In [ ]:
# Construir términos del GAM
terms_list = []
for i, feat in enumerate(feature_cols):
    if feat in binary_feats:
        terms_list.append(l(i))
    else:
        terms_list.append(s(i, n_splines=N_SPLINES, spline_order=SPLINE_ORDER))

terms_formula = terms_list[0]
for t in terms_list[1:]:
    terms_formula = terms_formula + t

n_smooth = sum(1 for f in feature_cols if f not in set(binary_feats))
n_linear = len(binary_feats)
print(f'Fórmula GAM construida:')
print(f'  Términos suaves   (s): {n_smooth}  → {n_smooth * N_SPLINES} parámetros')
print(f'  Términos lineales (l): {n_linear}')
print(f'  Total términos       : {len(terms_list)}')

In [ ]:
print('=' * 65)
print('5-FOLD STRATIFIEDKFOLD CV — Selección de Lambda')
print('=' * 65)
print(f'  N_FOLDS   : {N_FOLDS}')
print(f'  LAM_GRID  : {len(LAM_GRID)} valores')
print(f'  Muestra   : {len(y):,} filas (completa)')
print(f'  Train/fold: ~{int(len(y)*0.8):,} filas')
print(f'  Total fits: {N_FOLDS * len(LAM_GRID)}')
print('─' * 65)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_results = {}   # lam → mean_auc

t_cv = time.time()
for lam_idx, lam in enumerate(LAM_GRID):
    t_lam     = time.time()
    fold_aucs = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        gam_cv = LogisticGAM(terms_formula, lam=lam)
        gam_cv.fit(X_scaled[tr_idx], y[tr_idx])
        prob_val = gam_cv.predict_proba(X_scaled[val_idx])
        fold_aucs.append(roc_auc_score(y[val_idx], prob_val))
    cv_results[lam] = float(np.mean(fold_aucs))
    elapsed = time.time() - t_lam
    print(f'  [{lam_idx+1}/{len(LAM_GRID)}] lam={lam:9.5f}  '
          f'CV AUC={cv_results[lam]:.5f}  std={np.std(fold_aucs):.5f}  '
          f'({elapsed/60:.1f} min)')

best_lam    = max(cv_results, key=cv_results.get)
best_cv_auc = cv_results[best_lam]
print('─' * 65)
print(f'★  best_lam    = {best_lam:.5f}')
print(f'★  best_cv_auc = {best_cv_auc:.5f}')
print(f'   Tiempo total: {(time.time()-t_cv)/60:.1f} min')
print('=' * 65)

In [ ]:
lams     = list(cv_results.keys())
aucs     = [cv_results[l] for l in lams]
best_idx = aucs.index(max(aucs))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.log10(lams), aucs, 'o-', color='#e74c3c', lw=2, ms=7, label='CV AUC (5-fold)')
ax.scatter([np.log10(lams[best_idx])], [aucs[best_idx]],
           color='#3498db', s=140, zorder=5,
           label=f'Óptimo: λ={best_lam:.4f}  AUC={best_cv_auc:.5f}')
ax.axvline(np.log10(best_lam), color='#3498db', ls='--', lw=1.2, alpha=0.6)
ax.set_xlabel('log₁₀(λ)')
ax.set_ylabel('CV AUC (5-fold)')
ax.set_title('Selección de Lambda — LogisticGAM (5-fold Stratified KFold)')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_gam_cv_lambda.png', dpi=120)
plt.show()
print('Gráfico guardado: logit_gam_cv_lambda.png')

In [ ]:
print(f'Refitteando GAM final en muestra completa ({len(y):,} filas)...')
print(f'  lambda = {best_lam:.5f}')
t0 = time.time()
gam_final = LogisticGAM(terms_formula, lam=best_lam)
gam_final.fit(X_scaled, y)
print(f'  Refit completado ✓  ({time.time()-t0:.0f}s)')

y_prob_train = gam_final.predict_proba(X_scaled)
train_auc    = roc_auc_score(y, y_prob_train)
print(f'  Train AUC (full) : {train_auc:.5f}')
print(f'  CV AUC (5-fold)  : {best_cv_auc:.5f}')
print(f'  Gap              : {train_auc - best_cv_auc:.5f}')

In [ ]:
metrics = compute_metrics(y, y_prob_train, label='LogisticGAM')
metrics['CV_AUC']   = round(best_cv_auc, 5)
metrics['best_lam'] = round(float(best_lam), 6)
print('─' * 55)
for k, v in metrics.items():
    print(f'  {k:<15}: {v}')
print('─' * 55)
pd.DataFrame([metrics])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
fpr, tpr, _ = roc_curve(y, y_prob_train)
ax.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'LogisticGAM (AUC={train_auc:.5f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR (1 - Especificidad)')
ax.set_ylabel('TPR (Sensibilidad)')
ax.set_title('ROC Curve — Logit GAM (pyGAM)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_gam_roc.png', dpi=120)
plt.show()
print('Gráfico guardado: logit_gam_roc.png')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_train[y == val]
    kde   = gaussian_kde(probs, bw_method=0.1)
    xs    = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)
ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — LogisticGAM')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_gam_scores.png', dpi=120)
plt.show()
print('Gráfico guardado: logit_gam_scores.png')

In [ ]:
# Top 6 features continuas por AUC univariado
univ_auc = [(roc_auc_score(y, X_scaled[:, i]), i, feature_cols[i])
            for i in cont_idx]
univ_auc.sort(reverse=True)
top6 = univ_auc[:6]

print('Top 6 features continuas por AUC univariado:')
for auc_val, idx, name in top6:
    print(f'  {name:<40} AUC={auc_val:.4f}')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (_, feat_idx, feat_name) in zip(axes.flat, top6):
    XX          = gam_final.generate_X_grid(term=feat_idx)
    pdep, confi = gam_final.partial_dependence(term=feat_idx, X=XX, width=0.95)
    ax.plot(XX[:, feat_idx], pdep, color='#e74c3c', lw=2)
    ax.fill_between(XX[:, feat_idx], confi[:, 0], confi[:, 1],
                    alpha=0.2, color='#e74c3c')
    ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    ax.set_title(feat_name, fontsize=10)
    ax.set_xlabel(feat_name, fontsize=8)
    ax.set_ylabel('g(x)', fontsize=8)
plt.suptitle('Efectos suaves — LogisticGAM (top 6 features)', fontsize=13)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_gam_partial_dep.png', dpi=120)
plt.show()
print('Gráfico guardado: logit_gam_partial_dep.png')

## Guardar Artefactos

In [ ]:
# ── Modelo ─────────────────────────────────────────────────────────────────────────
path_pkl = MODEL_DIR / 'pygam_cloud_best.pkl'
joblib.dump({
    'gam'         : gam_final,
    'scaler'      : scaler,
    'imputer'     : imputer,
    'feature_cols': feature_cols,
    'lam'         : float(best_lam)
}, path_pkl)

# ── Metadata ────────────────────────────────────────────────────────────────────
metadata = {
    'model_type'     : 'logit_gam',
    'best_lam'       : float(best_lam),
    'best_cv_auc'    : round(best_cv_auc, 6),
    'train_auc'      : round(train_auc, 6),
    'n_folds'        : N_FOLDS,
    'lam_grid'       : [float(l) for l in LAM_GRID],
    'cv_results'     : {str(round(float(l), 6)): round(v, 6) for l, v in cv_results.items()},
    'n_splines'      : N_SPLINES,
    'spline_order'   : SPLINE_ORDER,
    'feature_cols'   : feature_cols,
    'n_train'        : int(len(y)),
    'sklearn_version': sklearn.__version__,
    'timestamp'      : pd.Timestamp.now().isoformat()
}

meta_path = MODEL_DIR / 'logit_splines_cloud_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

artefacts = [path_pkl, meta_path]
print('=' * 65)
print('ARTEFACTOS GUARDADOS')
print('=' * 65)
for p in artefacts:
    size_str = f'({p.stat().st_size/1e6:.2f} MB)' if p.exists() else ''
    print(f'  {p.name:<50} {size_str}')
print('=' * 65)
print('\n>>> Descargar desde el Output tab de Kaggle:')
for p in artefacts:
    print(f'    - {p.name}')
print('\n>>> Luego correr localmente: logit_cloud_predict.ipynb')

## Resumen Final — Logit GAM Cloud

In [ ]:
print('=' * 65)
print('LOGIT GAM CLOUD — RESUMEN FINAL')
print('=' * 65)
print(f'  Modelo        : LogisticGAM (pyGAM)')
print(f'  Lambda óptimo : {best_lam:.6f}')
print(f'  n_splines     : {N_SPLINES}  |  spline_order: {SPLINE_ORDER}')
print(f'  n_folds       : {N_FOLDS}')
print(f'  CV AUC        : {best_cv_auc:.5f}')
print(f'  Train AUC     : {train_auc:.5f}')
print(f'  Gap           : {train_auc - best_cv_auc:.5f}')
print('─' * 65)
print(f'  sklearn   : {sklearn.__version__}')
print(f'  Python    : {platform.python_version()}')
print(f'  Timestamp : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 65)